## OpenAI API_KET 세팅

In [2]:
# imports

import os
from dotenv import load_dotenv

In [3]:
# Initialization

load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

OpenAI API Key exists and begins sk-proj-


# Create speech

https://developers.openai.com/api/reference/python/resources/audio/subresources/speech/methods/create


In [1]:
from pathlib import Path
from openai import OpenAI
from IPython.display import Audio, display

client = OpenAI()

def talker(message):
    # 음성 파일 저장 경로 설정
    # Path(__file__): 현재 파일 위치 기준 (노트북에서는 현재 디렉토리)
    speech_file_path = Path("speech.mp3")

    # 음성 생성 및 스트리밍으로 파일 저장
    # with_streaming_response: 대용량 오디오도 메모리 효율적으로 처리
    with client.audio.speech.with_streaming_response.create(
        model="gpt-4o-mini-tts",        # TTS 모델 (tts-1 / tts-1-hd / gpt-4o-mini-tts)
        voice="onyx",                   # 목소리: alloy / ash / ballad / coral / echo /
                                        #         fable / onyx / nova / sage / shimmer /
                                        #         verse / marin / cedar
        input=message,                  # 읽어줄 텍스트 (최대 4096자)
        instructions="친근하고 따뜻한 목소리로 천천히 읽어주세요.",  # 음성 스타일 지시 (선택)
        response_format="mp3",          # 출력 형식: mp3 / opus / aac / flac / wav / pcm
        speed=1.0                       # 재생 속도: 0.25 ~ 4.0 (기본값 1.0)
    ) as response:
        # 스트리밍 응답을 MP3 파일로 저장
        response.stream_to_file(speech_file_path)

    # 주피터 노트북에서 오디오 바로 재생
    display(Audio(str(speech_file_path), autoplay=True))

    # 저장된 파일의 바이너리 데이터 반환
    return speech_file_path.read_bytes()


# 실행 예시
audio_data = talker("안녕하세요! 오늘도 좋은 하루 보내세요.")
print("음성 생성 완료!")

음성 생성 완료!


# Audio output from model

https://developers.openai.com/api/docs/guides/audio?example=audio-out


In [6]:
import base64
from openai import OpenAI
from IPython.display import Audio, display

client = OpenAI()

# Chat Completions API로 텍스트 + 오디오 동시 응답 요청
# - modalities: 응답 형식 지정 (text=텍스트, audio=오디오 동시 반환)
# - audio: 음성 설정 (voice=목소리 종류, format=오디오 파일 형식)
completion = client.chat.completions.create(
    model="gpt-audio-mini",                     # 오디오 출력 지원 모델
    modalities=["text", "audio"],               # 텍스트 + 오디오 동시 반환
    audio={
        "voice": "alloy",                       # 목소리: alloy / ash / ballad / coral /
                                                #         echo / sage / shimmer / verse
        "format": "wav"                         # 오디오 형식: wav / mp3 / opus / flac
    },
    messages=[
        {
            "role": "user",
            "content": "골든 리트리버는 좋은 가족 강아지인가요?"  # 한글 질문
        }
    ]
)

# 응답 전체 구조 출력 (디버깅용)
# - message.content: 텍스트 응답
# - message.audio.data: base64 인코딩된 오디오 데이터
# - message.audio.transcript: 오디오의 텍스트 스크립트
print(completion.choices[0])

# 텍스트 응답 출력
print("\n[텍스트 응답]")
print(completion.choices[0].message.audio.transcript)

# base64 인코딩된 오디오 데이터 → 바이너리 바이트로 디코딩
wav_bytes = base64.b64decode(completion.choices[0].message.audio.data)

# 디코딩된 바이트를 WAV 파일로 저장
# "wb" 모드: 바이너리 쓰기 (w=write, b=binary)
with open("dog.wav", "wb") as f:
    f.write(wav_bytes)

print("\n[오디오 파일 저장 완료]: dog.wav")

# 주피터 노트북에서 오디오 바로 재생
# autoplay=True: 셀 실행 시 자동 재생
display(Audio("dog.wav", autoplay=True))

Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=ChatCompletionAudio(id='audio_6a0d6503dd8c81918965c999de189fcf', data='...', expires_at=1779266323, transcript='네, 골든 리트리버는 정말 훌륭한 가족 강아지로 유명합니다. 매우 친근하고, 지능도 높고, 다루기 쉬운 편이라 아이들이 있어도 잘 어울리는 경우가 많죠. 단, 충분한 운동량과 사랑이 필요하기 때문에 그런 점만 잘 챙기면 아주 좋은 반려견이 될 수 있어요.'), function_call=None, tool_calls=None))

[텍스트 응답]
네, 골든 리트리버는 정말 훌륭한 가족 강아지로 유명합니다. 매우 친근하고, 지능도 높고, 다루기 쉬운 편이라 아이들이 있어도 잘 어울리는 경우가 많죠. 단, 충분한 운동량과 사랑이 필요하기 때문에 그런 점만 잘 챙기면 아주 좋은 반려견이 될 수 있어요.

[오디오 파일 저장 완료]: dog.wav


# Audio input to Model

https://cdn.openai.com/API/docs/audio/alloy.wav


https://developers.openai.com/api/docs/guides/audio?example=audio-in


In [8]:
import base64
import requests
from openai import OpenAI
from IPython.display import Audio, display

client = OpenAI()

# ① 오디오 파일 URL에서 다운로드
url = "https://cdn.openai.com/API/docs/audio/alloy.wav"
response = requests.get(url)
response.raise_for_status()             # HTTP 오류 발생 시 예외 처리 (4xx, 5xx)

# 다운로드한 WAV 파일을 노트북에서 바로 재생 (입력 오디오 확인용)
print("[입력 오디오]")
display(Audio(response.content, autoplay=True))

# ② 오디오 바이너리 데이터 → base64 문자열로 인코딩
# API는 바이너리를 직접 받지 않고 base64 문자열로 전송해야 함
wav_data = response.content
encoded_string = base64.b64encode(wav_data).decode('utf-8')

# ③ Chat Completions API로 오디오 분석 요청
# - 텍스트 질문 + 오디오 파일을 함께 전송 (멀티모달 입력)
# - modalities: 텍스트 + 오디오 동시 반환 요청
completion = client.chat.completions.create(
    model="gpt-audio",                          # 오디오 입출력 지원 모델
    modalities=["text", "audio"],               # 응답 형식: 텍스트 + 오디오 동시 반환
    audio={
        "voice": "alloy",                       # 응답 음성 종류
        "format": "wav"                         # 응답 오디오 형식
    },
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "이 녹음에는 무엇이 담겨 있나요?"   # 오디오 내용 질문
                },
                {
                    "type": "input_audio",                      # 오디오 입력 타입
                    "input_audio": {
                        "data": encoded_string,                 # base64 인코딩된 오디오 데이터
                        "format": "wav"                         # 입력 오디오 형식
                    }
                }
            ]
        },
    ]
)

# ④ 응답 메시지 전체 구조 출력 (디버깅용)
print("\n[응답 메시지 전체 구조]")
print(completion.choices[0].message)

# 텍스트 응답(스크립트) 출력
print("\n[텍스트 응답]")
print(completion.choices[0].message.audio.transcript)

# ⑤ 응답 오디오 데이터 디코딩 후 파일 저장
wav_bytes = base64.b64decode(completion.choices[0].message.audio.data)
with open("response.wav", "wb") as f:
    f.write(wav_bytes)

print("\n[응답 오디오 저장 완료]: response.wav")

# 응답 오디오를 노트북에서 바로 재생
print("[응답 오디오]")
display(Audio("response.wav", autoplay=True))

[입력 오디오]



[응답 메시지 전체 구조]
ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=ChatCompletionAudio(id='audio_6a0d65889fa4819185477f4886e2dafc', data='...', expires_at=1779266456, transcript='이 녹음에는 동쪽에서 해가 뜨고 서쪽으로 진다는 간단한 사실이 담겨 있어요. 그리고 이 사실은 인간이 수천 년 동안 관찰해 온 거라고 설명하고 있습니다.'), function_call=None, tool_calls=None)

[텍스트 응답]
이 녹음에는 동쪽에서 해가 뜨고 서쪽으로 진다는 간단한 사실이 담겨 있어요. 그리고 이 사실은 인간이 수천 년 동안 관찰해 온 거라고 설명하고 있습니다.

[응답 오디오 저장 완료]: response.wav
[응답 오디오]


# Multimodel & Gradio

1. 초기 설정 

In [9]:
# imports
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

# Some imports for handling images
import base64
from io import BytesIO
from PIL import Image

In [10]:
# Initialization

load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

OpenAI API Key exists and begins sk-proj-


In [14]:
MODEL = "gpt-4.1-mini"
openai = OpenAI()

2. 시스템 메시지 

In [15]:
# 시스템 메시지: FlightAI 항공사 AI 어시스턴트 역할 정의
system_message = """
당신은 FlightAI라는 항공사의 친절한 AI 어시스턴트입니다.
짧고 정중한 답변을 제공하며, 1~2문장을 넘지 않도록 하세요.
항상 정확한 정보를 제공하고, 모르는 경우 솔직히 말씀해 주세요.
대화 중 특정 도시가 언급되면 자연스럽게 그 도시에 대해 이야기해 주세요.
"""

3. 도시 감지 툴 정의 

In [16]:
def detect_city(city):
    """도시 이름을 반환하는 툴 실행 함수"""
    return city

In [17]:
# 도시 감지 툴 정의
# - 대화 중 언급된 도시를 추출하는 함수 스펙 정의
# - OpenAI가 이 툴을 호출하면 handle_tool_calls_and_return_cities()가 실행됨
tools = [
    {
        "type": "function",
        "function": {
            "name": "detect_city",
            "description": "대화에서 여행 목적지로 언급된 도시 이름을 감지합니다. 사용자가 특정 도시에 대해 묻거나 언급할 때 호출하세요.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "감지된 도시 이름 (예: 파리, 도쿄, 뉴욕)"
                    }
                },
                "required": ["city"]
            }
        }
    }
]

4. 툴 호출 처리 함수

In [18]:
def handle_tool_calls_and_return_cities(message):
    """
    모델이 tool_calls를 요청했을 때 처리하는 함수
    - tool_calls: 모델이 도시를 감지했을 때 호출하는 함수 목록
    - 각 툴 호출 결과를 수집하여 cities 리스트로 반환
    """
    import json
    responses = []
    cities = []

    for tool_call in message.tool_calls:
        # 툴 호출 인자 파싱 (JSON 문자열 → 딕셔너리)
        arguments = json.loads(tool_call.function.arguments)

        # detect_city 툴이 호출된 경우 도시 이름 추출
        if tool_call.function.name == "detect_city":
            city = arguments.get("city")
            cities.append(city)

            # 툴 실행 결과를 messages에 추가 (API 요구사항)
            responses.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": f"{city} 도시가 감지되었습니다."
            })

    return responses, cities

5. 이미지 생성 함수 

In [19]:
def artist(city):
    """
    도시를 표현하는 팝아트 스타일 이미지 생성
    - gpt-image-1-mini 모델 사용 (dall-e-3 서비스 종료)
    - base64 형식으로 응답 받아 PIL Image로 변환
    """
    image_response = openai.images.generate(
        model="gpt-image-1-mini",                   # dall-e-3 → gpt-image-1-mini
        prompt=f"{city} 여행을 표현한 이미지. {city}의 유명 관광지와 고유한 특색을 생동감 넘치는 팝아트 스타일로 그려주세요.",
        size="1024x1024",                           # 이미지 크기
        quality="low",          # low 품질로 생성 속도 향상 (low < medium < high)
        n=1,                                        # 생성할 이미지 수
        # response_format 제거 — gpt-image 계열은 항상 b64_json 반환
    )
    # base64 디코딩 → 바이너리 → PIL Image 객체로 변환
    image_base64 = image_response.data[0].b64_json
    image_data = base64.b64decode(image_base64)
    return Image.open(BytesIO(image_data))

6. 음성 변환 함수 

In [20]:
def talker(message):
    """
    텍스트를 음성으로 변환 (TTS)
    - gpt-4o-mini-tts 모델 사용
    - 스트리밍 방식으로 음성 데이터 반환
    """
    with openai.audio.speech.with_streaming_response.create(
        model="gpt-4o-mini-tts",
        voice="onyx",           # 목소리: alloy / coral / onyx / nova / shimmer 등
        input=message,
        response_format="wav"   # 오디오 형식
    ) as response:
        # 스트리밍 응답을 바이트로 읽어 반환
        return b"".join(response.iter_bytes())


7. 메인 채팅 함수 


In [21]:
def chat(history):
    """
    채팅 메인 함수
    1. 대화 히스토리 + 시스템 메시지로 API 호출
    2. 도시 감지 툴 호출 처리 (tool_calls 루프)
    3. 최종 텍스트 응답 → 음성 변환
    4. 도시 감지 시 → 이미지 생성
    """
    # 대화 히스토리를 API 형식으로 변환
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history

    # 도시 감지 툴을 포함하여 첫 번째 API 호출
    response = openai.chat.completions.create(
        model=MODEL,
        messages=messages,
        tools=tools,            # 도시 감지 툴 등록
        tool_choice="auto"      # 모델이 자동으로 툴 사용 여부 결정
    )

    cities = []
    image = None

    # 모델이 툴 호출을 요청하는 동안 반복 처리
    # finish_reason == "tool_calls": 모델이 도시를 감지하여 툴 호출 요청
    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        # 툴 호출 처리 및 감지된 도시 목록 반환
        responses, cities = handle_tool_calls_and_return_cities(message)
        messages.append(message)
        messages.extend(responses)
        # 툴 결과를 포함하여 다시 API 호출
        response = openai.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools,
            tool_choice="auto"
        )

    # 최종 텍스트 응답 추출
    reply = response.choices[0].message.content
    history += [{"role": "assistant", "content": reply}]

    # 텍스트 응답 → 음성 변환
    voice = talker(reply)

    # 도시가 감지된 경우 첫 번째 도시의 이미지 생성
    if cities:
        image = artist(cities[0])

    return history, voice, image
#end chat

In [22]:
def put_message_in_chatbot(message, history):
    """사용자 메시지를 챗봇 히스토리에 추가"""
    return "", history + [{"role": "user", "content": message}]

In [ ]:
# Gradio UI 구성 — gr.Blocks() 컨텍스트 안에 모든 컴포넌트와 이벤트를 정의
with gr.Blocks() as ui:

    # 1행: 채팅창(왼쪽) + 도시 이미지(오른쪽) 나란히 배치
    with gr.Row():
        # 대화 내용을 표시하는 채팅 컴포넌트
        # type="messages": {"role": ..., "content": ...} 형식 사용
        chatbot = gr.Chatbot(height=500, type="messages", label="FlightAI 채팅")

        # 도시 이미지 출력 컴포넌트
        # interactive=False: 사용자가 직접 이미지를 업로드/수정 불가
        image_output = gr.Image(height=500, interactive=False, label="도시 이미지")

    # 2행: 음성 응답 재생 바
    with gr.Row():
        # AI 응답 음성을 재생하는 오디오 컴포넌트
        # autoplay=True: 새 음성이 생성되면 자동으로 재생
        audio_output = gr.Audio(autoplay=True, label="음성 응답")

    # 3행: 사용자 텍스트 입력창
    with gr.Row():
        message = gr.Textbox(label="AI 어시스턴트와 대화하세요 (예: 도쿄 여행 정보 알려줘):")

    # 이벤트 연결: 사용자가 Enter 또는 전송 버튼을 누르면 순차 실행
    message.submit(
        # 1단계: 사용자 메시지를 즉시 채팅창에 표시
        # inputs : 입력창 텍스트 + 현재 채팅 히스토리
        # outputs: 입력창 초기화("") + 업데이트된 채팅 히스토리
        put_message_in_chatbot,
        inputs=[message, chatbot],
        outputs=[message, chatbot]
    ).then(
        # 2단계: AI 응답 생성 (텍스트 + 음성 + 이미지)
        # 1단계 완료 후 자동으로 실행됨
        # inputs : 업데이트된 채팅 히스토리
        # outputs: 채팅 히스토리 + 음성 데이터 + 도시 이미지
        chat,
        inputs=chatbot,
        outputs=[chatbot, audio_output, image_output]
    )

# Gradio 앱 실행
# inbrowser=True: 실행 시 브라우저 자동으로 열기
# auth 제거: 로그인 없이 바로 접속 가능
ui.launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


# end